# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of Analysis

One row represents the daily performance of one content item for one client on one report date.

### Time Window

The analysis uses data from March 2026 (month = 2026-03).

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features
- gsc_clicks
- gsc_impressions
- gsc_sum_position
- sessions
- scroll_events

### Label
- Search ranking score (or relevance score)

### Context
- report_date
- client_hash_id
- content_hash_id

### Excluded
- URL (excluded because it is not available in this table and could introduce data leakage if used).

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [10]:
con.sql("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
LIMIT 5;
""")

┌─────────────┬─────────────────────────┬──────────────────────────┐
│ report_date │     client_hash_id      │     content_hash_id      │
│    date     │         varchar         │         varchar          │
├─────────────┼─────────────────────────┼──────────────────────────┤
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_b7e512995f79d5a6 │
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_05597932fe4da067 │
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_7a105f548d9c6916 │
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_905aa32a0230694e │
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_a3ea9792f793ec72 │
└─────────────┴─────────────────────────┴──────────────────────────┘

In [11]:
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_day,
    MAX(report_date) AS last_day
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
);
""")

┌────────────┬────────────┬────────────┐
│ total_rows │ first_day  │  last_day  │
│   int64    │    date    │    date    │
├────────────┼────────────┼────────────┤
│    9841378 │ 2026-03-01 │ 2026-03-31 │
└────────────┴────────────┴────────────┘

In [12]:
con.sql("""
SELECT
    COUNT(*) AS available_rows
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_data_available IS TRUE;
""")

┌────────────────┐
│ available_rows │
│     int64      │
├────────────────┤
│        3611061 │
└────────────────┘

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data Limits

- The dataset contains historical search performance only.
- It cannot explain why users clicked or ignored a result.
- Some clients have shorter history than others (unbalanced history).
- Early records may contain only Google Search Console data because GA4 was not yet available for all clients.
- The dataset is useful for decision support, not for proving causation.

In [4]:
con.sql("""
SELECT *
FROM glob('hf://datasets/FlyRank/internship-warehouse/**')
""")

┌────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│                                                  file                                                  │
│                                                varchar                                                 │
├────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ hf://datasets/FlyRank/internship-warehouse/.gitattributes                                              │
│ hf://datasets/FlyRank/internship-warehouse/README.md                                                   │
│ hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet                                         │
│ hf://datasets/FlyRank/internship-warehouse/dim_content.parquet                                         │
│ hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-01/data_0.parquet │
│ hf://datasets/FlyRank/internship-wa

In [5]:
con.sql("""
DESCRIBE
SELECT *
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""")

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

In [6]:
con.sql("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
LIMIT 5;
""")

┌─────────────┬─────────────────────────┬──────────────────────────┐
│ report_date │     client_hash_id      │     content_hash_id      │
│    date     │         varchar         │         varchar          │
├─────────────┼─────────────────────────┼──────────────────────────┤
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_b7e512995f79d5a6 │
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_05597932fe4da067 │
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_7a105f548d9c6916 │
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_905aa32a0230694e │
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_a3ea9792f793ec72 │
└─────────────┴─────────────────────────┴──────────────────────────┘

In [7]:
con.sql("""
SELECT
COUNT(*) AS total_rows,
MIN(report_date) AS first_day,
MAX(report_date) AS last_day
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
);
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────┬────────────┐
│ total_rows │ first_day  │  last_day  │
│   int64    │    date    │    date    │
├────────────┼────────────┼────────────┤
│    9841378 │ 2026-03-01 │ 2026-03-31 │
└────────────┴────────────┴────────────┘

In [13]:
result = con.sql("""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_day,
    MAX(report_date) AS last_day
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
);
""")

result

┌────────────┬────────────┬────────────┐
│ total_rows │ first_day  │  last_day  │
│   int64    │    date    │    date    │
├────────────┼────────────┼────────────┤
│    9841378 │ 2026-03-01 │ 2026-03-31 │
└────────────┴────────────┴────────────┘

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.